# 03 · Feature engineering cho nhận dạng video âm nhạc

Nghiên cứu ngày 2026-09-11; chưa thay classifier production hoặc nhãn trong DB. [Tổng hợp phương pháp](../docs/references/music-feature-engineering.md).

Mục tiêu là tìm feature giúp phân biệt **nội dung chính là nhạc**, không phải dự đoán video người dùng thích. Số liệu proxy Topic/library không phải precision/recall. Không tải metadata/audio/model; đọc SQLite ở chế độ read-only, output cá nhân chỉ vào artifacts.

Tham khảo: [YouTube 2016: candidate/ranking](https://research.google.com/pubs/archive/45530.pdf), [Spotify: chuẩn hóa hành vi cá nhân](https://research.atspotify.com/2018/7/understanding-and-evaluating-user-satisfaction-with-music-discovery), [PISA/Deezer, RecSys 2024: phiên và lặp lại](https://arxiv.org/abs/2408.16578). Đây là các phương pháp công bố, không khẳng định kiến trúc production hiện tại của các nền tảng.

In [ ]:
import os, json, re, sqlite3, hashlib, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from auralytica.classification import TOPIC, MUSIC, TALK, RULE_VERSION
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
DATABASE = Path(os.environ.get('AURALYTICA_EDA_DB', str(Path.home()/'.local/share/auralytica/library.sqlite3'))).expanduser().resolve()
TIMEZONE = os.environ.get('AURALYTICA_EDA_TIMEZONE', 'Asia/Bangkok')
assert DATABASE.is_file(), 'Chọn AURALYTICA_EDA_DB trỏ tới thư viện đã import.'
with sqlite3.connect(DATABASE.as_uri()+'?mode=ro', uri=True) as db:
    db.execute('BEGIN')
    active = db.execute("SELECT value FROM settings WHERE key='active_import'").fetchone()
    assert active, 'Chưa có active import.'
    events = pd.read_sql_query('SELECT source_row,video_id,watched_at FROM watch_events WHERE import_id=? AND video_id IS NOT NULL', db, params=(active[0],))
    videos = pd.read_sql_query('SELECT * FROM videos WHERE id IN (SELECT video_id FROM watch_events WHERE import_id=?)', db, params=(active[0],)).rename(columns={'id':'video_id'})
    source_hash = db.execute('SELECT source_hash FROM imports WHERE id=?', (active[0],)).fetchone()[0]
    db.rollback()
snapshot = hashlib.sha256((events.to_json()+videos.to_json()).encode()).hexdigest()[:12]
analysis_code = ''.join(''.join(c.get('source', [])) for c in json.loads((ROOT/'notebooks/03_music_feature_engineering.ipynb').read_text())['cells'] if c['cell_type']=='code')
feature_version = 'fe-v1-'+hashlib.sha256(analysis_code.encode()).hexdigest()[:8]
OUT = ROOT/'artifacts/notebook-runs/03_music_features'/snapshot/feature_version
OUT.mkdir(parents=True, exist_ok=True)
events['time'] = pd.to_datetime(events.watched_at, utc=True, errors='coerce')
quality = {'events':len(events), 'videos':len(videos), 'missing_timestamps':int(events.time.isna().sum()),
           'duplicate_video_timestamp':int(events.dropna(subset=['time']).duplicated(['video_id','time']).sum()),
           'timezone':TIMEZONE, 'source_hash':source_hash, 'snapshot':snapshot}
display(pd.Series(quality))

## 1. Lượt xem cá nhân: tần suất, độ bền và cơ hội quan sát

Giữ raw watch_count giống app. Loại trùng ID/timestamp **chỉ khi tính feature thời gian**, không xóa sự kiện nguồn. Ngày tính theo timezone cấu hình. Dữ liệu thiếu timestamp không biến thành số lần quay lại bằng 0.

`observed_day_coverage` = số ngày xem / số ngày từ lần thấy đầu tới cuối export; khoảng này bị giới hạn bởi export, không phải tuổi video. `same_day_concentration` giúp phân biệt nhiều lượt dồn một ngày với quay lại lâu dài; cả hai vẫn chỉ là giả thuyết. Percentile là so với các video của chính người dùng, không so tổng view YouTube.

In [ ]:
f = videos.set_index('video_id').copy()
f['watch_count'] = events.groupby('video_id').size()
f['log_watch_count'] = np.log1p(f.watch_count)
f['personal_repeat_percentile'] = f.watch_count.rank(method='average', pct=True)
timed = events.dropna(subset=['time']).drop_duplicates(['video_id','time']).sort_values(['time','source_row']).copy()
timed['day'] = timed.time.dt.tz_convert(TIMEZONE).dt.date
f['distinct_watch_times'] = timed.groupby('video_id').size()
f['timestamp_coverage'] = f.distinct_watch_times.fillna(0)/f.watch_count
f['watch_days'] = timed.groupby('video_id').day.nunique()
f['span_days'] = timed.groupby('video_id').time.agg(lambda x:(x.max()-x.min()).total_seconds()/86400)
f['max_watches_one_day'] = timed.groupby(['video_id','day']).size().groupby('video_id').max()
f['same_day_concentration'] = f.max_watches_one_day/f.distinct_watch_times
first = timed.groupby('video_id').time.min()
end_day = timed.time.max().tz_convert(TIMEZONE).date()
f['observed_days'] = first.dt.tz_convert(TIMEZONE).dt.date.map(lambda day: (end_day-day).days+1)
f['observed_day_coverage'] = f.watch_days/f.observed_days
f['days_since_last_watch'] = (timed.time.max()-timed.groupby('video_id').time.max()).dt.total_seconds()/86400
gaps = timed.groupby('video_id').time.diff().dt.total_seconds()/86400
f['median_rewatch_gap_days'] = gaps.groupby(timed.video_id).median()
f['observed_weeks'] = timed.groupby('video_id').day.agg(lambda x:len({d.isocalendar()[:2] for d in x}))
assert f.watch_count.sum() == len(events)
assert (f.watch_days.dropna() <= f.distinct_watch_times.dropna()).all()
assert f.observed_day_coverage.dropna().between(0,1).all()
display(f[['watch_count','watch_days','same_day_concentration','span_days']].describe())

## 2. Nội dung và bằng chứng kênh

Tách các family tín hiệu, không gộp về một boolean music_hint. Các regex mở rộng bên dưới là **feature nghiên cứu**, không phải quyết định nhận nhạc. Cụm mạnh vẫn có phản ví dụ reaction/tutorial; không gán kênh nhiều nội dung thành kênh nhạc chỉ từ một video.

Proxy seed dùng Topic/library sau khi xét dấu hiệu nói chuyện và Shorts, không dùng user_group hoặc auto_group. Tỷ lệ seed của kênh loại chính video đang xét (leave-one-out) và luôn kèm số mẫu. Tên kênh không có ID không được gộp vào một kênh giả. Nhãn kiểm chứng không được đưa trở lại feature.

In [ ]:
def norm(s): return unicodedata.normalize('NFKC', str(s or '')).casefold()
def flag(series, pattern): return series.map(lambda x:bool(re.search(pattern,x,re.I)))
f['title_norm'] = f.title.map(norm)
f['topic_seed'] = f.channel_name.fillna('').map(lambda x:bool(TOPIC.search(x)))
meta = f.metadata_json.map(json.loads)
f['library_seed'] = meta.map(lambda m:m.get('music_library') is True)
f['shorts_explicit'] = meta.map(lambda m:m.get('takeout_shorts_url') is True)
f['talk_existing'] = f.title.map(lambda x:bool(TALK.search(x)))
f['music_existing'] = f.title.map(lambda x:bool(MUSIC.search(x)))
f['song_format'] = flag(f.title_norm, r'(?<![a-z0-9])(?:ost|amv|instrumental|soundtrack)(?![a-z0-9])|歌ってみた|カバー|노래')
f['performance_phrase'] = flag(f.title_norm, r'music\s+video|official\s+audio|live\s+symphony|orchestr(?:a|al)|symphon(?:y|ic)|nhạc')
f['version_terms'] = flag(f.title_norm, r'(?<![a-z0-9])(?:remix|cover|slowed|reverb|lofi|lo-fi|nightcore)(?![a-z0-9])')
f['talk_format'] = f.talk_existing | flag(f.title_norm, r'behind.the.scenes|making.of|review|analysis|giải thích|phân tích|bình luận')
f['artist_title_shape'] = flag(f.title_norm, r'^.{2,60}\s[-–—]\s.{2,120}$')
f['proxy_seed'] = (f.topic_seed | f.library_seed) & ~f.shorts_explicit & ~f.talk_format
f['content_candidate'] = f.music_existing | f.song_format | f.performance_phrase | f.version_terms
channel = f.channel_key.replace('', pd.NA)
f['channel_video_count'] = f.groupby(channel).title.transform('size')
f['channel_other_seed_share'] = (f.groupby(channel).proxy_seed.transform('sum')-f.proxy_seed.astype(int))/(f.channel_video_count-1).replace(0,np.nan)
f['channel_personal_watch_share'] = f.watch_count / f.groupby(channel).watch_count.transform('sum')
f['repeat_content_interaction'] = f.log_watch_count * f.content_candidate.astype(int)
f['effective_group'] = f.user_group.fillna(f.auto_group)
# Assertions on Unicode handling, not claims about classification quality.
assert bool(re.search(r'(?<![a-z0-9])ost(?![a-z0-9])', norm('稲妻OSTイメージ')))
assert f.index.is_unique
assert f.channel_other_seed_share.dropna().between(0,1).all()

## 3. Phiên xem suy đoán và ngữ cảnh gần video

Takeout không có session ID. Thử ngưỡng khoảng cách 15/30/60 phút và so độ nhạy; không coi 30 phút là chân lý. Không biến khoảng cách hai sự kiện thành watch duration/completion/skip.

`neighbor_seed_share` dùng video khác nằm ngay trước/sau trong ngưỡng phiên. Loại chính video đó và trường hợp timestamp bằng nhau; mẫu số được lưu. Đây là feature hồi cứu để lọc export đã có, không dùng kết quả sau thời điểm dự đoán nếu sau này làm model online.

In [ ]:
session_summary=[]
for minutes in [15,30,60]:
    session = timed.time.diff().gt(pd.Timedelta(minutes=minutes)).cumsum()
    f[f'inferred_sessions_{minutes}m'] = timed.assign(session=session).groupby('video_id').session.nunique()
    session_summary.append({'gap_minutes':minutes,'sessions_total':int(session.nunique()),
                            'videos_in_3plus_sessions':int((f[f'inferred_sessions_{minutes}m']>=3).sum())})
neighbor_parts=[]
for shift in [-1,1]:
    other_id=timed.video_id.shift(shift)
    delta=(timed.time-timed.time.shift(shift)).abs()
    valid=delta.gt(pd.Timedelta(0)) & delta.le(pd.Timedelta(minutes=30)) & other_id.ne(timed.video_id)
    neighbor_parts.append(pd.DataFrame({'video_id':timed.video_id[valid], 'seed':other_id[valid].map(f.proxy_seed).astype(int)}))
neighbors=pd.concat(neighbor_parts)
f['neighbor_count'] = neighbors.groupby('video_id').size().reindex(f.index).fillna(0).astype(int)
f['neighbor_seed_share'] = neighbors.groupby('video_id').seed.mean()
assert f.neighbor_seed_share.dropna().between(0,1).all()
display(pd.DataFrame(session_summary))

## 4. Insight có thể đo ngay; chưa phải xác suất nhạc

Các bảng đo **độ phủ feature**, phân phối và số ca rule-v1 chưa chọn. So sánh seed với non-seed chỉ để khám phá; non-seed chứa nhiều nhạc nên không phải lớp âm. Không dùng chính tỷ lệ Topic/library để tuyên bố feature phân loại tốt.

Không công bố precision/recall cho đến khi có nhãn video độc lập. Nhạc nghe một lần vẫn phải được giữ trong thiết kế và mẫu kiểm chứng.

In [ ]:
bins=pd.cut(f.watch_count,[0,1,2,4,9,np.inf],labels=['1','2','3–4','5–9','10+'])
counts=f.groupby(bins,observed=True).agg(videos=('title','size'),baseline_music=('effective_group',lambda x:(x=='music').sum()),
                                    proxy_seeds=('proxy_seed','sum'),median_watch_days=('watch_days','median'))
display(counts)
rest=f.effective_group.eq('rest')
insights={'video_count':len(f),'watch_events':len(events),'baseline_music':int((~rest).sum()),
          'rest_repeat_3plus':int((rest & f.watch_count.ge(3)).sum()),
          'rest_revisit_3plus_days':int((rest & f.watch_days.ge(3)).sum()),
          'rest_existing_music_terms':int((rest & f.music_existing).sum()),
          'rest_expanded_content':int((rest & f.content_candidate).sum()),
          'rest_new_content_without_old_terms':int((rest & f.content_candidate & ~f.music_existing).sum()),
          'rest_repeat_3plus_and_content':int((rest & f.watch_count.ge(3) & f.content_candidate).sum()),
          'rest_repeat_3plus_no_content':int((rest & f.watch_count.ge(3) & ~f.content_candidate).sum()),
          'rest_content_talk_conflict':int((rest & f.content_candidate & f.talk_format).sum()),
          'seed_seen_once':int((f.proxy_seed & f.watch_count.eq(1)).sum()),
          'missing_timestamps':quality['missing_timestamps']}
display(pd.Series(insights))
fig,ax=plt.subplots(figsize=(8,4))
counts[['videos','baseline_music']].plot.bar(ax=ax,logy=True)
ax.set(xlabel='Personal watch-count bucket',ylabel='Videos (log scale)',title='Coverage of current rule; not accuracy')
fig.tight_layout();fig.savefig(OUT/'repeat_coverage.png',dpi=160)
plt.show()
f.reset_index().to_csv(OUT/'video_features.csv',index=False)
(OUT/'summary.json').write_text(json.dumps({'quality':quality,'insights':insights,'sessions':session_summary,'rule_version':RULE_VERSION,'feature_version':feature_version},ensure_ascii=False,indent=2))

## 5. Mẫu khám phá và tập đánh giá

Không đọc nhãn holdout cũ. Nếu dùng nguồn khác, phải cấu hình lại EVAL_FILES đúng export. Các ID eval được loại khỏi sample khám phá (ngoại trừ những ca đã lộ nhãn trong hội thoại cần loại khỏi eval khi nghiệm thu). Các CSV mới không ghi đè nhãn khi chạy lại.

Mẫu khám phá phủ bốn tình huống: nghe lại nhưng không có từ khóa; cụm nhạc nhưng chỉ xem một lần; xung đột nội dung; chưa có tín hiệu. Giới hạn 3 video/kênh giúp tránh mẫu bị một kênh chi phối. Mẫu này dùng tìm insight/tune, **không dùng ước lượng precision/recall toàn thư viện**. Không hiển thị tên/nhãn của holdout trong notebook này.

In [ ]:
EVAL_FILES = list((ROOT/'artifacts/notebook-runs/01_takeout_eda').glob('*/signals-v2/random_review.csv'))
held_out=set()
for path in EVAL_FILES:
    columns=pd.read_csv(path,nrows=0).columns
    key='video_id' if 'video_id' in columns else 'id'
    held_out.update(pd.read_csv(path,usecols=[key])[key].astype(str))
pool=f.loc[~f.index.isin(held_out)].copy()
strata={
 'repeat_without_content':pool.watch_count.ge(3)&~pool.content_candidate,
 'single_view_content':pool.watch_count.eq(1)&pool.content_candidate&~pool.proxy_seed,
 'content_talk_conflict':pool.content_candidate&pool.talk_format,
 'weak_or_missing_signals':~pool.content_candidate&~pool.proxy_seed&pool.watch_count.le(2),
}
parts=[];selected=set()
for name,mask in strata.items():
    candidates=pool.loc[mask & ~pool.index.isin(selected)].sample(frac=1,random_state=20260911)
    keys=candidates.channel_key.fillna(pd.Series(candidates.index,index=candidates.index))
    sample=candidates.groupby(keys,sort=False).head(3).head(40).copy()
    sample['stratum']=name;parts.append(sample);selected.update(sample.index)
review=pd.concat(parts).reset_index()
review['url']='https://www.youtube.com/watch?v='+review.video_id
review['manual_label']='';review['notes']=''
review_path=OUT/'discovery_review.csv'
if not review_path.exists(): review.to_csv(review_path,index=False)
assert not set(review.video_id)&held_out
print({'reserved_eval_ids':len(held_out),'discovery_rows':len(review),'output':str(OUT)})
display(review[['title','channel_name','watch_count','watch_days','inferred_sessions_30m','stratum','url']].head(12))

## 6. Xem lại phương thức hiện tại và thí nghiệm tiếp theo

1. Giữ baseline rules-v1 để so sánh; chưa thay nhóm trong ứng dụng từ feature nghiên cứu.
2. Đánh giá riêng hai mục tiêu: **tự chọn đúng nhạc** (precision/recall) và **tìm nhạc nhanh khi duyệt** (music@K, số nhạc còn bỏ sót, số thao tác). Candidate discovery có thể rộng, quyết định auto-select cần ngưỡng kiểm chứng riêng.
3. Ablation trên cùng split: nội dung → thêm recurrence → thêm kênh/ngữ cảnh → thêm metadata cache. Chỉ thêm feature nếu cải thiện trên nhãn validation, không chỉ khiến nhiều video chuyển trái hơn.
4. Khi có đủ nhãn, so rule kết hợp với logistic regression hoặc cây nhỏ; giữ tập kiểm tra cuối và kiểm tra split theo channel để phát hiện học thuộc kênh. Không ép model phức tạp khi mới có vài nhãn.
5. Metadata thiếu là missing, không là bằng chứng non-music. Category Music/artist/track có thể hữu ích nhưng game có BGM cũng có metadata nhạc. Lấy metadata thử nghiệm nhỏ sau khi biết nhóm thiếu bằng chứng.
6. Audio là phương án sau cùng cho ca giá trị cao chưa rõ: YAMNet cho sự kiện âm thanh theo cửa sổ, không trực tiếp trả nhãn “video có nội dung chính là nhạc”; chi phí lấy audio và false positive BGM phải đo riêng.

Cần báo confusion matrix, số mẫu và độ bất định; với sampling phân tầng phải dùng trọng số lấy mẫu hoặc eval ngẫu nhiên độc lập. Không dùng auto_group làm y, không lan truyền nhãn người dùng từ test vào feature kênh, không tune rồi đo trên cùng 5 video trong ảnh.